In [60]:
import pandas as pd
import numpy as np
import glob
import os
from typing import Dict, List, Tuple
import re

def extract_final_architecture_string(filepath: str) -> str:
    """Extract the final architecture string from architecture history file"""
    base_name = os.path.basename(filepath).replace('_architecture_history.txt', '')
    prefix = base_name.split('_')[0]
    last_number = prefix.split('.')[-1]
    final_architecture = []
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
        final_section_start = -1
        for i, line in enumerate(lines):
            if line.strip() == "Final Architecture:":
                final_section_start = i
                break
                
        if final_section_start == -1:
            raise ValueError("No 'Final Architecture:' section found in file")
            
        for line in lines[final_section_start + 1:]:
            line = line.strip()
            if not line:
                break
                
            if "Layer" in line and ":" in line:
                parts = line.split(":")
                if len(parts) != 2:
                    continue
                    
                neurons_info = parts[1].strip()
                if "Not active" in neurons_info:
                    final_architecture.append('n')
                else:
                    try:
                        num_neurons = int(neurons_info.split()[0])
                        final_architecture.append(str(num_neurons))
                    except (ValueError, IndexError):
                        continue
    
    architecture_str = '_'.join(final_architecture)
    result = f"{prefix}.{last_number}.{architecture_str}"
    return result

def process_csv(file_path: str) -> pd.DataFrame:
    """Process a CSV file containing validation accuracy data"""
    df = pd.read_csv(file_path)
    df['value'] = df['value'].apply(lambda x: eval(x)[0])
    df = df.drop_duplicates(subset=['step'], keep='last')
    return df

def calculate_metrics(df: pd.DataFrame) -> Dict[str, float]:
    """Calculate various performance metrics from the DataFrame"""
    final_accuracy = df['value'].tail(10).mean()
    
    # Convergence Time (steps to reach within 1% of final value)
    final_val = df['value'].iloc[-1]
    if 'loss' in directory_path.lower():
        # For loss: when it drops to within 1% of final loss
        convergence_threshold = final_val * 1.01
        convergence_step = df[df['value'] <= convergence_threshold]['step'].iloc[0]
    else:
        # For accuracy: when it reaches 99% of final accuracy
        convergence_threshold = final_val * 0.99
        convergence_step = df[df['value'] >= convergence_threshold]['step'].iloc[0]
    
    # Training Stability (CV)
    cv = df['value'].std() / df['value'].mean()
    
    # Computing Time
    total_time = df['wall_time'].iloc[-1] - df['wall_time'].iloc[0]
    avg_time_per_step = total_time / len(df)
    
    return {
        'final_accuracy': final_accuracy,
        'convergence_time': convergence_step,
        'stability_cv': cv,
        'total_training_time': total_time,
        'avg_time_per_step': avg_time_per_step
    }

def aggregate_metrics(metrics_list: List[Dict[str, float]]) -> Dict[str, Tuple[float, float]]:
    """Calculate mean and std for each metric across multiple runs"""
    all_metrics = {}
    for metric in metrics_list[0].keys():
        values = [m[metric] for m in metrics_list]
        mean_val = np.mean(values)
        std_val = np.std(values)
        all_metrics[metric] = (mean_val, std_val)
    return all_metrics

def format_aggregate_metrics(senn_metrics: Dict[str, Tuple[float, float]], 
                           mlp_metrics: Dict[str, Tuple[float, float]], tag: str) -> None:
    """Print formatted comparison of metrics"""
    print(f"\nPerformance Metrics Comparison ({tag}):")
    print("-" * 80)
    print(f"{'Metric':<30} {'SENN':^22} {'MLP':^22}")
    print("-" * 80)
    
    # Determine metric name based on directory path
    metric_name = 'Loss' if 'loss' in directory_path.lower() else 'Accuracy'
    
    metrics_to_format = {
        f'Final {metric_name}': 'final_accuracy',
        'Convergence Time (steps)': 'convergence_time',
        'Training Stability (CV)': 'stability_cv',
        'Total Training Time (s)': 'total_training_time',
        'Avg Time per Step (s)': 'avg_time_per_step'
    }
    
    for display_name, metric_key in metrics_to_format.items():
        senn_mean, senn_std = senn_metrics[metric_key]
        mlp_mean, mlp_std = mlp_metrics[metric_key]
        
        senn_str = f"{senn_mean:.3f} ± {senn_std:.3f}"
        mlp_str = f"{mlp_mean:.3f} ± {mlp_std:.3f}"
        print(f"{display_name:<30} {senn_str:<22} {mlp_str:<22}")

def process_directory(directory_path: str) -> None:
    """Process all validation accuracy files in directory"""
    tag = directory_path.split('/')[-1]
    csv_files = glob.glob(os.path.join(directory_path, f'*_{tag}.csv'))
    
    if not csv_files:
        # print(f"No CSV files found in {directory_path}")
        return
    
    experiment_name = os.path.basename(os.path.dirname(directory_path))
    arch_base_path = os.path.join('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures', experiment_name)
    
    senn_metrics = []
    mlp_metrics = []
    base_to_arch = {}
    
    # First identify base experiments and their final architectures
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        parts = base_name.split('.')
        
        if len(parts) == 2:  # Base SENN experiment
            arch_file = os.path.join(arch_base_path, f'{base_name}_architecture_history.txt')
            if os.path.exists(arch_file):
                try:
                    final_arch = extract_final_architecture_string(arch_file)
                    base_to_arch[base_name] = final_arch
                    # Process SENN metrics
                    df = process_csv(csv_file)
                    senn_metrics.append(calculate_metrics(df))
                except Exception as e:
                    print(f"Error processing SENN file {base_name}: {e}")
    
    # Now process final architecture MLPs
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        if base_name in base_to_arch.values():  # Only process final architectures
            try:
                df = process_csv(csv_file)
                mlp_metrics.append(calculate_metrics(df))
            except Exception as e:
                print(f"Error processing MLP file {base_name}: {e}")
    
    if not senn_metrics or not mlp_metrics:
        print("No valid files found for either SENN or MLP")
        return
    
    # Calculate aggregate metrics
    senn_aggregate = aggregate_metrics(senn_metrics)
    mlp_aggregate = aggregate_metrics(mlp_metrics)
    
    # Print results
    # print(f"\nProcessed {len(senn_metrics)} SENN runs and {len(mlp_metrics)} MLP runs")
    format_aggregate_metrics(senn_aggregate, mlp_aggregate, tag=tag)

In [61]:
for ex in range(1, 3):
    print(f"\nProcessing Experiment {ex}...")
    for tag in ["loss", "validation loss", "training accuracy", "validation accuracy"]:
        directory_path = f"/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment{ex}/{tag}"
        process_directory(directory_path)


Processing Experiment 1...

Performance Metrics Comparison (loss):
--------------------------------------------------------------------------------
Metric                                  SENN                   MLP          
--------------------------------------------------------------------------------
Final Loss                     0.000 ± 0.000          0.000 ± 0.000         
Convergence Time (steps)       1951.633 ± 43.334      1930.367 ± 80.783     
Training Stability (CV)        16.818 ± 3.165         16.919 ± 3.395        
Total Training Time (s)        4064.238 ± 127.444     83.701 ± 6.232        
Avg Time per Step (s)          2.032 ± 0.064          0.042 ± 0.003         

Performance Metrics Comparison (validation loss):
--------------------------------------------------------------------------------
Metric                                  SENN                   MLP          
--------------------------------------------------------------------------------
Final Loss        

In [62]:
def print_value(directory_path: str, time_step: int) -> None:
    """Process all validation accuracy files and print mean ± std at given time step"""
    tag = directory_path.split('/')[-1]
    csv_files = glob.glob(os.path.join(directory_path, f'*_{tag}.csv'))
    
    if not csv_files:
        print(f"No CSV files found in {directory_path}")
        return
    
    experiment_name = os.path.basename(os.path.dirname(directory_path))
    arch_base_path = os.path.join('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures', experiment_name)
    
    senn_values = []
    mlp_values = []
    base_to_arch = {}
    
    # Process SENN files
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        parts = base_name.split('.')
        
        if len(parts) == 2:  # Base SENN experiment
            arch_file = os.path.join(arch_base_path, f'{base_name}_architecture_history.txt')
            if os.path.exists(arch_file):
                try:
                    final_arch = extract_final_architecture_string(arch_file)
                    base_to_arch[base_name] = final_arch
                    df = process_csv(csv_file)
                    value = df[df['step'] == time_step]['value'].values[0]
                    senn_values.append(value)
                except Exception as e:
                    print(f"Error processing SENN file {base_name}: {e}")
    
    # Process MLP files
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        if base_name in base_to_arch.values():
            try:
                df = process_csv(csv_file)
                value = df[df['step'] == time_step]['value'].values[0]
                mlp_values.append(value)
            except Exception as e:
                print(f"Error processing MLP file {base_name}: {e}")
    
    if senn_values and mlp_values:
        senn_mean, senn_std = np.mean(senn_values), np.std(senn_values)
        mlp_mean, mlp_std = np.mean(mlp_values), np.std(mlp_values)
        print(f"\nValues at step {time_step}:")
        print("-" * 80)
        print(f"{'Model':<30} {'Value':^22}")
        print("-" * 80)
        print(f"{'SENN':<30} {senn_mean:.3f} ± {senn_std:.3f}")
        print(f"{'MLP':<30} {mlp_mean:.3f} ± {mlp_std:.3f}")

In [63]:
tag = "validation loss"
ex = 2
time_step = 34
directory_path = f"/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment{ex}/{tag}"
# Call the function with the given directory_path and time_step
print_value(directory_path, time_step=time_step)


Values at step 34:
--------------------------------------------------------------------------------
Model                                  Value         
--------------------------------------------------------------------------------
SENN                           0.415 ± 0.011
MLP                            0.422 ± 0.055


## Expansion Analysis

In [64]:
import glob
import os
import numpy as np
from typing import List, Dict, Tuple
import re

def parse_architecture(section: List[str]) -> Dict[int, int]:
    """Parse a single architecture section to get layer sizes."""
    layer_sizes = {}
    for line in section:
        if "Layer" in line and ":" in line:
            try:
                layer_num = int(line.split("Layer")[1].split(":")[0].strip())
                if "Not active" in line:
                    layer_sizes[layer_num] = 0
                else:
                    neurons = int(line.split("neurons")[0].split(":")[-1].strip())
                    layer_sizes[layer_num] = neurons
            except (ValueError, IndexError):
                continue
    return layer_sizes

def determine_expansion_type(prev_arch: Dict[int, int], curr_arch: Dict[int, int]) -> str:
    """Determine if expansion was layer addition or neuron addition."""
    active_layers_prev = sum(1 for size in prev_arch.values() if size > 0)
    active_layers_curr = sum(1 for size in curr_arch.values() if size > 0)
    
    if active_layers_curr > active_layers_prev:
        return "layer"
    return "neuron"

def extract_expansion_details(filepath: str) -> List[Tuple[int, str]]:
    """Extract epochs and types of architecture changes."""
    expansions = []
    architectures = []
    current_section = []
    current_epoch = None
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
        
        for line in lines:
            line = line.strip()
            
            if not line:
                if current_section:
                    arch = parse_architecture(current_section)
                    if current_epoch is not None:
                        architectures.append((current_epoch, arch))
                    current_section = []
                continue
                
            if "Initial Architecture:" in line or "Final Architecture:" in line:
                current_epoch = 0
                current_section = []
            elif "Architecture after epoch" in line:
                try:
                    current_epoch = int(line.split("epoch")[1].split(":")[0].strip())
                    current_section = []
                except (ValueError, IndexError):
                    continue
            
            current_section.append(line)
            
        # Process last section if exists
        if current_section:
            arch = parse_architecture(current_section)
            if current_epoch is not None:
                architectures.append((current_epoch, arch))
    
    # Determine expansion types by comparing consecutive architectures
    # Skip the initial architecture (index 0) when it has epoch 0
    start_idx = 1 if architectures and architectures[0][0] == 0 else 0
    
    for i in range(start_idx, len(architectures)):
        epoch = architectures[i][0]
        prev_arch = architectures[i-1][1]
        curr_arch = architectures[i][1]
        exp_type = determine_expansion_type(prev_arch, curr_arch)
        if epoch > 0:  # Only include non-zero epochs
            expansions.append((epoch, exp_type))
    
    return sorted(expansions, key=lambda x: x[0])

def calculate_parameters(architecture: Dict[int, int], input_size: int) -> int:
    """Calculate total parameters for MLP architecture."""
    active_layers = [(k, v) for k, v in sorted(architecture.items()) if v > 0]
    if not active_layers:
        return 0
        
    total_params = 0
    prev_size = input_size
    
    for _, curr_size in active_layers:
        # Weight parameters: prev_size * curr_size
        # Bias parameters: curr_size
        params = (prev_size * curr_size) + curr_size
        total_params += params
        prev_size = curr_size
        
    return total_params

def analyze_expansion_timings(directory_path: str, experiment: str = "ex1") -> None:
    """Analyze expansion timings and types across all architecture files."""
    input_size = 1 if experiment == "ex1" else 2  # ex1: regression, ex2: half moon
    """Analyze expansion timings and types across all architecture files."""
    arch_files = glob.glob(os.path.join(directory_path, '*architecture_history.txt'))
    
    if not arch_files:
        print("No architecture history files found")
        return
        
    expansion_timings: Dict[int, List[Tuple[int, str]]] = {}
    
    for file_path in arch_files:
        try:
            expansions = extract_expansion_details(file_path)
            
            for i, (epoch, exp_type) in enumerate(expansions):
                if i not in expansion_timings:
                    expansion_timings[i] = []
                expansion_timings[i].append((epoch, exp_type))
                
        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")
            continue
    
    # Track architectures at each expansion point
    expansion_architectures: Dict[int, List[Dict[int, int]]] = {}
    
    for file_path in arch_files:
        current_section = []
        current_epoch = None
        current_arch = None
        last_expansion_num = -1
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
            
            for line in lines:
                line = line.strip()
                
                if not line:
                    if current_section and current_epoch is not None:
                        arch = parse_architecture(current_section)
                        if current_arch is None or arch != current_arch:
                            current_arch = arch
                            last_expansion_num += 1
                            if last_expansion_num not in expansion_architectures:
                                expansion_architectures[last_expansion_num] = []
                            expansion_architectures[last_expansion_num].append(arch)
                    current_section = []
                    continue
                
                if "Architecture after epoch" in line:
                    try:
                        current_epoch = int(line.split("epoch")[1].split(":")[0].strip())
                        current_section = []
                    except (ValueError, IndexError):
                        continue
                        
                current_section.append(line)
    
    print("\nExpansion Timing Analysis:")
    print("-" * 100)
    print(f"Analyzed {len(arch_files)} architecture files")
    print("-" * 100)
    print(f"{'Expansion':<10} {'Mean Epoch':<12} {'Std Dev':<12} {'Count':<8} {'Neuron Add %':<12} {'Mean Params':<12}")
    print("-" * 100)
    
    for exp_num in sorted(expansion_timings.keys()):
        expansions = expansion_timings[exp_num]
        epochs = [e[0] for e in expansions]
        mean_epoch = np.mean(epochs)
        std_epoch = np.std(epochs)
        count = len(epochs)
        
        # Calculate percentage of neuron additions
        neuron_adds = sum(1 for _, type_ in expansions if type_ == "neuron")
        neuron_percent = (neuron_adds / count) * 100
        
        # Calculate mean parameters for this expansion
        if exp_num in expansion_architectures:
            params = [calculate_parameters(arch, input_size) for arch in expansion_architectures[exp_num]]
            mean_params = np.mean(params)
        else:
            mean_params = 0
            
        print(f"{f'#{exp_num+1}':<10} {f'{mean_epoch:.1f}':<12} {f'{std_epoch:.1f}':<12} "
              f"{count:<8} {f'{neuron_percent:.1f}%':<12} {f'{mean_params:.1f}':<12}")
    
    # Additional statistics
    all_files_expansions = [len(extract_expansion_details(f)) for f in arch_files]
    avg_expansions = np.mean(all_files_expansions)
    std_expansions = np.std(all_files_expansions)
    
    print("-" * 100)
    print(f"Average number of expansions per network: {avg_expansions:.1f} ± {std_expansions:.1f}")

In [65]:
ex = 2
directory_path = f"/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures/experiment{ex}"

analyze_expansion_timings(directory_path, experiment=ex)


Expansion Timing Analysis:
----------------------------------------------------------------------------------------------------
Analyzed 30 architecture files
----------------------------------------------------------------------------------------------------
Expansion  Mean Epoch   Std Dev      Count    Neuron Add % Mean Params 
----------------------------------------------------------------------------------------------------
#1         66.0         12.0         30       0.0%         17.3        
#2         188.6        101.1        28       50.0%        29.2        
#3         284.7        63.2         23       17.4%        47.9        
#4         309.8        53.0         4        75.0%        46.8        
#5         390.0        0.0          1        100.0%       47.0        
----------------------------------------------------------------------------------------------------
Average number of expansions per network: 2.9 ± 0.8


In [31]:
import pandas as pd
import numpy as np

def calculate_expansion_improvements(csv_file_path: str, arch_file_path: str, window_size: int = 5) -> List[Tuple[int, float]]:
    """
    Calculate performance improvement achieved by each expansion using smoothed values.
    Uses mean of window_size epochs before each measurement point.
    """
    
    def extract_expansion_epochs(filepath: str) -> List[int]:
        """Extract epochs of architecture changes."""
        epochs = []
        with open(filepath, 'r') as f:
            lines = f.readlines()
            for line in lines:
                if "Architecture after epoch" in line:
                    try:
                        epoch = int(line.split("epoch")[1].split(":")[0].strip())
                        epochs.append(epoch)
                    except (ValueError, IndexError):
                        continue
        return epochs

    def process_csv(file_path: str) -> pd.DataFrame:
        """Process CSV file and return DataFrame with step and value columns."""
        df = pd.read_csv(file_path)
        df['value'] = df['value'].apply(lambda x: float(x.strip('[]')))
        return df
    
    def get_smoothed_loss(df: pd.DataFrame, epoch: int, window_size: int, before: bool = True) -> float:
        """
        Get smoothed loss value around the given epoch.
        If before=True, takes window before epoch, otherwise after epoch.
        """
        if before:
            mask = (df['step'] <= epoch) & (df['step'] > epoch - window_size)
        else:
            mask = (df['step'] >= epoch) & (df['step'] < epoch + window_size)
        
        values = df[mask]['value'].values
        if len(values) == 0:
            return df[df['step'] == epoch]['value'].iloc[0]
        return np.mean(values)

    def calculate_improvement(initial_loss: float, final_loss: float) -> float:
        """Calculate improvement percentage with robust handling of small numbers."""
        if np.isclose(initial_loss, 0, atol=1e-15):
            return 0.0
        
        improvement = np.float64(initial_loss - final_loss) / np.float64(initial_loss) * 100
        
        if np.isnan(improvement) or np.isinf(improvement):
            return 0.0
            
        return float(improvement)

    # Extract expansion epochs
    expansion_epochs = extract_expansion_epochs(arch_file_path)
    
    # Process CSV file
    performance_df = process_csv(csv_file_path)
    
    # Calculate improvements
    improvements = []
    raw_values = []  # Store raw values for debugging
    
    # For all expansions except the last one
    for i in range(len(expansion_epochs)-1):
        curr_epoch = expansion_epochs[i]
        next_epoch = expansion_epochs[i+1]
        
        try:
            # Get smoothed loss values
            curr_loss = get_smoothed_loss(performance_df, curr_epoch, window_size, before=True)
            next_loss = get_smoothed_loss(performance_df, next_epoch, window_size, before=True)
            
            # Store raw values for debugging
            raw_values.append((curr_loss, next_loss))
            
            # Calculate improvement
            improvement = calculate_improvement(curr_loss, next_loss)
            improvements.append((curr_epoch, improvement))
            
        except (IndexError, ValueError) as e:
            print(f"Warning: Error processing epoch {curr_epoch}: {e}")
            improvements.append((curr_epoch, 0.0))
    
    # Handle last expansion
    try:
        last_epoch = expansion_epochs[-1]
        last_epoch_loss = get_smoothed_loss(performance_df, last_epoch, window_size, before=True)
        final_loss = get_smoothed_loss(performance_df, performance_df['step'].max(), window_size, before=True)
        final_improvement = calculate_improvement(last_epoch_loss, final_loss)
        improvements.append((last_epoch, final_improvement))
        raw_values.append((last_epoch_loss, final_loss))
    except (IndexError, ValueError) as e:
        print(f"Warning: Error processing final epoch: {e}")
        improvements.append((last_epoch, 0.0))
    
    # Print raw values for verification
    # print("\nRaw smoothed loss values used for calculations:")
    # for i, (curr_loss, next_loss) in enumerate(raw_values):
    #     epoch = expansion_epochs[i]
    #     # print(f"Epoch {epoch:4d}: {curr_loss:.16e} → {next_loss:.16e}")
    
    return improvements


In [57]:
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict
from pathlib import Path
import re

def calculate_mean_expansion_improvements(
    base_csv_dir: str,
    base_arch_dir: str,
    experiment: str,
    window_size: int = 5,
    tag: str = "loss"
) -> Dict[int, float]:
    """
    Calculate mean performance improvement per expansion number across multiple seeds
    for a single experiment.
    
    Parameters:
    -----------
    base_csv_dir : str
        Base directory containing loss CSV files
    base_arch_dir : str
        Base directory containing architecture history files
    experiment : str
        Experiment number to process
    window_size : int
        Window size for smoothing, default=5
        
    Returns:
    --------
    Dict[int, float]
        Dictionary mapping expansion number (0-based) to mean improvement percentage
    """
    
    # Store improvements by expansion number
    all_improvements: Dict[int, List[float]] = {}
    
    # Construct base paths for the experiment
    exp_csv_dir = Path(base_csv_dir) / f"experiment{experiment}" / f"{tag}"
    exp_arch_dir = Path(base_arch_dir) / f"experiment{experiment}"
    
    # Find all seed files for this experiment using strict pattern
    pattern = re.compile(f"ex{experiment}\\.[0-9]+_{tag}\\.csv$")
    csv_files = [f for f in exp_csv_dir.iterdir() if pattern.match(f.name)]
    
    if not csv_files:
        raise ValueError(f"No matching files found for experiment {experiment}")
    
    # print(f"Found {len(csv_files)} seed files for experiment {experiment}")
    
    for csv_file in sorted(csv_files):
        # Extract seed number from filename
        # Filename format: ex{experiment}.{seed}_loss.csv
        seed = csv_file.stem.split('_')[0].split('.')[1]
        
        # Construct corresponding architecture file path
        arch_file = exp_arch_dir / f"ex{experiment}.{seed}_architecture_history.txt"
        
        # try:
        # Get improvements for this seed
        improvements = calculate_expansion_improvements(str(csv_file), str(arch_file), window_size)
        
        # Store improvements by expansion number (0-based index)
        for idx, (_, improvement) in enumerate(improvements):
            if idx not in all_improvements:
                all_improvements[idx] = []
            all_improvements[idx].append(improvement)
                
        # except Exception as e:
        #     print(f"Warning: Error processing experiment {experiment} seed {seed}: {e}")
        #     continue
    
    # Calculate mean improvement for each expansion number
    mean_improvements = {}
    stats = {}  # Store additional statistics
    
    for expansion_num, improvements in all_improvements.items():
        if improvements:  # Check if we have any valid improvements
            mean_improvements[expansion_num] = np.mean(improvements)
            stats[expansion_num] = {
                'mean': np.mean(improvements),
                'std': np.std(improvements),
                'median': np.median(improvements),
                'num_runs': len(improvements)
            }
        else:
            mean_improvements[expansion_num] = 0.0
            stats[expansion_num] = {
                'mean': 0.0,
                'std': 0.0,
                'median': 0.0,
                'num_runs': 0
            }
            
    return mean_improvements, stats

def print_expansion_statistics(stats: Dict[int, Dict], tag) -> None:
    """
    Print formatted statistics about improvements per expansion.
    """
    print(f"\nPerformance Improvements per Expansion ({tag}):")
    print("-" * 53)
    print(f"{'Expansion':^10} | {'Mean Improvement (%)':^20} | {'Std Dev':^15} |")
    print("-" * 53)
    
    for expansion_num, stat in sorted(stats.items()):
        print(f"{(expansion_num+1):^10} | {stat['mean']:^20.2f} | {stat['std']:^15.2f} |")

In [68]:
# Setup your directories
base_csv_dir = "/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports"
base_arch_dir = "/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures"

# Specify the experiment number you want to analyze
experiment = "2"
tag = "validation accuracy"

# Calculate mean improvements and statistics
mean_improvements, stats = calculate_mean_expansion_improvements(
    base_csv_dir,
    base_arch_dir,
    experiment,
    tag = tag
)

# Print the results
print_expansion_statistics(stats, tag)


Performance Improvements per Expansion (validation accuracy):
-----------------------------------------------------
Expansion  | Mean Improvement (%) |     Std Dev     |
-----------------------------------------------------
    1      |        -3.71         |      2.56       |
    2      |        -1.94         |      2.85       |
    3      |         0.78         |      1.09       |
    4      |         0.60         |      1.08       |
    5      |        -0.18         |      0.00       |


## Intermediate Arch Analysis

In [277]:
ex = "1"
seed = "10"
tag = "loss"
base_csv_dir = "/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports"
base_arch_dir = "/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures"

#return me all files that belong to seed
def get_seed_files(ex: int, seed: int, tag: str, base_csv_dir: str) -> List[str]:
    """Get all CSV files for a specific seed in an experiment."""
    exp_csv_dir = Path(base_csv_dir) / f"experiment{ex}" / f"{tag}"
    pattern = re.compile(f"ex{ex}\\.{seed}(?![0-9]).*_{tag}\\.csv$")
    csv_files = [f for f in exp_csv_dir.iterdir() if pattern.match(f.name)]
    return csv_files  

get_seed_files(ex, seed, tag, base_csv_dir)

[PosixPath('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment1/loss/ex1.10.10.3_1_loss.csv'),
 PosixPath('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment1/loss/ex1.10.10.11_1_loss.csv'),
 PosixPath('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment1/loss/ex1.10.10.9_1_loss.csv'),
 PosixPath('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment1/loss/ex1.10_loss.csv'),
 PosixPath('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment1/loss/ex1.10.10.8_1_loss.csv'),
 PosixPath('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment1/loss/ex1.10.10.1_1_loss.csv'),
 PosixPath('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment1/loss/ex1.10.10.6_1_loss.csv'),
 PosixPath('

In [278]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

def get_param_count(architecture, input_dim):
    """Calculate parameter count for a given architecture."""
    layers = [input_dim] + architecture  # Add input dimension
    total_params = 0
    for i in range(len(layers)-1):
        # Weight matrix parameters: current_layer * next_layer
        # Bias parameters: next_layer
        total_params += (layers[i] * layers[i+1]) + layers[i+1]
    return total_params

def parse_architecture(filename, tag):
    """Extract architecture from filename."""
    # For base file (e.g., ex1.10_loss.csv), return None
    if re.match(f'ex\d+\.\d+_{tag}\.csv$', str(filename).split('/')[-1]):
        return None
    
    # For experiment 2, format is like ex2.10.10.4_4_n_2_loss.csv -> [4,4,2]
    filename_str = str(filename)
    if 'ex2' in filename_str:
        match = re.search(f'\.(\d+)_(\d+)_n_(\d+)_{tag}\.csv$', filename_str)
        if match:
            return [int(match.group(1)), int(match.group(2)), int(match.group(3))]
    
    # For experiment 1, format is like ex1.10.10.5_1_loss.csv -> [5,1]
    match = re.search(f'\.(\d+(?:_\d+)*(?:_n_\d+)?)_{tag}\.csv$', filename_str)
    if match:
        parts = match.group(1).split('_')
        return [int(x) for x in parts]
    
    return None


def analyze_architectures(files, experiment, tag):
    # Find base file (pattern: ex{experiment}.{seed}_loss.csv)
    base_file = None
    for f in files:
        if re.match(f'ex{experiment}\\.\\d+_{tag}\\.csv$', str(f).split('/')[-1]):
            base_file = f
            break
    
    if not base_file:
        raise ValueError("Base file not found")
    
    # Read base file and get final performance
    base_df = pd.read_csv(base_file)
    base_final_loss = float(base_df['value'].iloc[-1].strip('[]'))
    threshold = base_final_loss * 1.05  # 95% performance = 105% of loss
    
    # Process each file
    valid_architectures = []
    for file in files:
        if file == base_file:
            continue
            
        df = pd.read_csv(file)
        final_loss = float(df['value'].iloc[-1].strip('[]'))
        
        if final_loss <= threshold:
            arch = parse_architecture(file, tag)
            if arch:  # Skip base file which has no architecture
                input_dim = 2 if experiment == "2" else 1
                params = get_param_count(arch, input_dim)
                valid_architectures.append((params, arch, file))
    
    if valid_architectures:
        # Sort by parameter count and get the smallest
        min_params, min_arch, min_file = min(valid_architectures)
        # print(f"Selected architecture: {min_arch}")
        # print(f"From file: {min_file}")
        return min_params
    
    return None


In [279]:
ex = "2"
base_csv_dir = "/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports"
base_arch_dir = "/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures"

for ex in ["1", "2"]:
    # List to store relative parameter counts
    if ex == "1":
        tag = "validation loss"
    else:   
        tag = "validation accuracy"
        
    relative_params_list = []
    counter = 0

    # Process all seeds
    for seed in range(0, 30):
        try:
            files = get_seed_files(ex, str(seed), tag, base_csv_dir)
            min_params = analyze_architectures(files, ex, tag)

            final = extract_final_architecture_string(f"{base_arch_dir}/experiment{ex}/ex{ex}.{seed}_architecture_history.txt")
            files = f"{base_csv_dir}/experiment{ex}/{tag}/{final}_{tag}.csv"
            arch = parse_architecture(files, tag)
            max_params = get_param_count(arch, int(ex))

            relative_params = (min_params / max_params) * 100
            relative_params_list.append(relative_params)
        except:
            counter += 1
            continue    

    # Calculate statistics
    relative_params_array = np.array(relative_params_list)
    mean_relative = np.mean(relative_params_array)
    std_relative = np.std(relative_params_array)
    success_rate = 100 * (1 - counter/30)
    failure_rate = 100 * counter/30

    print("\n" + "="*50)
    print("Summary Statistics for Fixed MLPs")
    print("="*50)
    print(f"Mean Parameter Usage:      {mean_relative:>8.2f}% ± {std_relative:.2f}%")
    print(f"Successful Architectures:  {30-counter:>8d} ({success_rate:>6.2f}%)")
    print(f"Failed Architectures:      {counter:>8d} ({failure_rate:>6.2f}%)")
    print("="*50)


Summary Statistics for Fixed MLPs
Mean Parameter Usage:         47.59% ± 20.90%
Successful Architectures:        19 ( 63.33%)
Failed Architectures:            11 ( 36.67%)

Summary Statistics for Fixed MLPs
Mean Parameter Usage:         85.91% ± 17.29%
Successful Architectures:        28 ( 93.33%)
Failed Architectures:             2 (  6.67%)
